# MiniMax H3 Image-to-Video with Multiple LoRAs

Animate a local starting image with `checkpoints/minimaxH3_comfy_3086249.safetensors`, the included `vae/minimax_h3_video_vae_fp16.safetensors`, and a configurable stack of H3 LoRAs. The included `Minimax_H3_Unlocked_V2.1.safetensors` adapter starts at strength 1.0; adjust each adapter independently in `loras`.

This notebook uses the native ComfyUI H3 Python runtime directly in the kernel. Assets belong under `Models/Model_Families/MiniMax/`; input images, outputs, and `.env` paths resolve from the project root. The WAN notebook and application workflows remain separate.


## Settings

Set `input_image` to an existing local image and edit the prompts. Add more `{ "path": "...", "strength": 0.8 }` entries to `loras`, or use `[]` for the base model. LoRA paths resolve from `MiniMax/loras/`. Strength 0 skips an adapter. Use MiniMax H3 / FL2VA-compatible files and any required prompt triggers.

H3 requires dimensions divisible by 32 and frame counts `17*n + 5` at 24 fps. The default 124 frames lasts about 5.17 seconds. Sampling follows the official base workflow: 20 steps, `res_multistep`, `simple`, and `BasicGuider` (CFG 1). Distillation adapters need their own documented settings.


In [9]:
# Edit these settings, then run all remaining cells.
from pathlib import Path

project_root = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "Models" / "Model_Families").is_dir()), None,
)
if project_root is None:
    raise FileNotFoundError("Start the kernel inside the Image_Generator project.")
family_dir = project_root / "Models" / "Model_Families" / "MiniMax"
comfyui_dir = project_root / "build" / "ComfyUI"

# Relative asset paths use the family folders indicated below. Absolute paths work too.
checkpoint = "minimaxH3_comfy_3086249.safetensors"  # Under MiniMax/checkpoints/.
vae_weights = "vae/minimax_h3_video_vae_fp16.safetensors"  # Under MiniMax/.
text_encoder_weights = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"  # MiniMax/text_encoders/.
download_text_encoder = True  # Download this encoder from Comfy-Org/MiniMax-H3 if missing.

input_image = 'Images/input.jpg'
prompts = ["The camera slowly moves forward. A gentle breeze moves through the scene, cinematic lighting, smooth natural motion as this beautiful woman bends over deep in a seductive manner"]
width = 832
height = 480
preserve_aspect_ratio = True
num_frames = 124              # H3 uses 17*n + 5 frames; 124 is about 5.17 seconds at 24 fps.
fps = 24                     # H3's native rate; leave at 24 for correct model timing.
num_inference_steps = 20
sampler_name = "res_multistep"
noise_schedule = "simple"
# H3's official base workflow uses BasicGuider (CFG=1), with no negative prompt or second expert.
num_videos_per_prompt = 1
seed = None
enable_vae_tiling = True

# Add any number of MiniMax H3 / FL2VA-compatible ComfyUI LoRAs.
# Paths are relative to MiniMax/loras/ or absolute. Each strength is independent.
# The included H3 adapter is selected below. An empty list uses the base checkpoint.
# WAN, SDXL, and FLUX LoRAs cannot be reused here. Add required trigger words to prompts.
loras = [
    {"path": "Minimax_H3_Unlocked_V2.1.safetensors", "strength": 1.0},
    # {"path": "motion/example.safetensors", "strength": 0.8},
    # {"path": "style/example.safetensors", "strength": 0.5},
]
# Distilled/Turbo adapters also require their documented step and sampling settings.
output_dir = 'Videos'
show_videos = True


## One-time setup

Run the settings cell first, then this setup cell in your Thor-compatible CUDA/PyTorch kernel. Setup clones ComfyUI into the ignored `build/ComfyUI/` folder and installs its dependencies while constraining PyTorch and torchvision to their installed versions. An existing checkout is reused; it must include native H3 support. Git and FFmpeg are required (imageio-ffmpeg may provide FFmpeg).

Restart the kernel after installation. For normal use, run **settings → helpers → generation**, skipping setup. No ComfyUI server or custom nodes are needed.

The checkpoint contains the diffusion model only. H3 also needs its Qwen3-VL-32B text encoder. Supply one in `MiniMax/text_encoders/`, or let the generation cell download the selected compressed encoder from [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3) into `CACHE_LOCATION/hub/` on first use. This is a separate multi-GB download; `download_text_encoder=False` requires a local file. Optional `HUGGINGFACE_TOKEN` and `CACHE_LOCATION` come from the project `.env`.

Setup prints `pip check` diagnostics. On Linux aarch64, the specific `nvidia-cusparselt-cu13 0.8.1` SBSA wheel-tag warning is allowed only after a fresh process successfully imports PyTorch/torchvision, loads cuSPARSELt 0.8.1, and runs CUDA BF16 matrix multiplication. Other pip errors and failed runtime checks stop setup. This check does not test every CUDA operation or full video inference.


In [2]:
# Keep the CUDA-enabled PyTorch/torchvision builds supplied for JetPack/Thor.
from importlib.metadata import version
import subprocess
import sys

for package in ("torch", "torchvision"):
    print(f"Installed {package}: {version(package)}")

if not comfyui_dir.exists():
    comfyui_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(comfyui_dir),
    ], check=True)
if not (comfyui_dir / "comfy_extras" / "nodes_minimax_h3.py").is_file():
    raise RuntimeError(f"Use a current ComfyUI checkout with native H3 support: {comfyui_dir}")

# Constraints prevent pip from replacing the platform-specific CUDA wheels.
constraints = project_root / "build" / "minimax-torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")))
subprocess.run([
    sys.executable, "-m", "pip", "install", "-c", str(constraints),
    "-r", str(comfyui_dir / "requirements.txt"),
    "huggingface_hub", "python-dotenv", "imageio", "imageio-ffmpeg", "ipykernel",
], check=True)
def check_notebook_environment():
    """Keep pip checks strict, with a runtime-verified exception for Thor's SBSA tag."""
    import platform
    from importlib.metadata import distribution

    result = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
    )
    report = "\n".join(part.strip() for part in (result.stdout, result.stderr) if part.strip())
    print(report or f"pip check exited with status {result.returncode}.")
    if result.returncode == 0:
        return

    # This specific NVIDIA wheel uses an SBSA tag pip doesn't recognize on aarch64.
    # Do not suppress other packages, versions, platform errors, or dependency conflicts.
    known_tag_issue = (
        result.returncode == 1
        and platform.system() == "Linux"
        and platform.machine() == "aarch64"
        and report == "nvidia-cusparselt-cu13 0.8.1 is not supported on this platform"
    )
    if known_tag_issue:
        package = distribution("nvidia-cusparselt-cu13")
        tags = {line.removeprefix("Tag: ").strip()
                for line in (package.read_text("WHEEL") or "").splitlines() if line.startswith("Tag: ")}
        known_tag_issue = package.version == "0.8.1" and tags == {"py3-none-manylinux2014_sbsa"}
    if not known_tag_issue:
        raise RuntimeError("pip check failed; resolve the dependency/platform errors printed above before continuing.")

    # Probe in a fresh process so the notebook's old imports cannot mask a broken install.
    # A failed probe still stops setup; no CUDA packages are removed or replaced.
    probe = '''
import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; the cuSPARSELt platform warning cannot be bypassed.")
if not torch.backends.cusparselt.is_available() or torch.backends.cusparselt.version() != 801:
    raise RuntimeError("Expected cuSPARSELt 0.8.1 could not be loaded by PyTorch.")
with torch.inference_mode():
    matrix = torch.ones((32, 32), device="cuda", dtype=torch.bfloat16)
    product = matrix @ matrix
    torch.cuda.synchronize()
    torch.testing.assert_close(product, torch.full_like(product, 32))
print(f"CUDA BF16 check passed on {torch.cuda.get_device_name(0)}; cuSPARSELt 0.8.1 loaded.")
'''
    subprocess.run([sys.executable, "-c", probe], check=True)
    print("Continuing after verifying CUDA: the only pip check issue is the known cuSPARSELt SBSA wheel tag.")


check_notebook_environment()
print("Restart the kernel, then run the settings, helpers, and generation cells. Skip setup on normal reruns.")


Installed torch: 2.14.1+cu132
Installed torchvision: 0.29.1+cu132
nvidia-cusparselt-cu13 0.8.1 is not supported on this platform
CUDA BF16 check passed on NVIDIA Thor; cuSPARSELt 0.8.1 loaded.
Continuing after verifying CUDA: the only pip check issue is the known cuSPARSELt SBSA wheel tag.
Restart the kernel, then run the settings, helpers, and generation cells. Skip setup on normal reruns.


## Helpers and validation

Run this cell after settings. Local checkpoint/VAE metadata, paths, dimensions, frame count, and LoRA settings are checked before loading model weights. The runtime caches model components and rebuilds changed LoRA stacks from the original model so adapters do not accumulate on reruns.


In [3]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import secrets
import struct
import subprocess
import sys
from time import perf_counter
from uuid import uuid4


def resolve_path(value, root):
    if not isinstance(value, (str, Path)) or not str(value).strip():
        raise ValueError("Expected a nonempty local path.")
    path = Path(value).expanduser()
    return (path if path.is_absolute() else root / path).resolve()


def fingerprint(path):
    stat = path.stat()
    return str(path), stat.st_size, stat.st_mtime_ns


def safetensors_header(path):
    # Read metadata only, including the extensionless VAE; never load tensors here.
    with path.open("rb") as stream:
        size_bytes = stream.read(8)
        if len(size_bytes) != 8:
            raise ValueError(f"Invalid safetensors file: {path}")
        size = struct.unpack("<Q", size_bytes)[0]
        if not 2 <= size <= min(path.stat().st_size - 8, 100_000_000):
            raise ValueError(f"Invalid safetensors header length: {path}")
        return json.loads(stream.read(size))


def validate_settings():
    if not isinstance(prompts, (list, tuple)) or not prompts or any(
        not isinstance(p, str) or not p.strip() for p in prompts
    ):
        raise ValueError("prompts must be a nonempty list of nonempty strings.")
    for name, value in {"width": width, "height": height}.items():
        if type(value) is not int or value < 32 or value % 32:
            raise ValueError(f"{name} must be a positive multiple of 32.")
    if type(num_frames) is not int or not 5 <= num_frames <= 3600 or num_frames % 17 != 5:
        raise ValueError("num_frames must be 17*n + 5, from 5 to 3600 (e.g. 107, 124, 243).")
    if type(fps) not in (int, float) or fps != 24:
        raise ValueError("MiniMax H3 uses native 24 fps; set fps = 24.")
    for name, value in {"num_inference_steps": num_inference_steps,
                        "num_videos_per_prompt": num_videos_per_prompt}.items():
        if type(value) is not int or value <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    if seed is not None and (type(seed) is not int or not 0 <= seed < 2**63):
        raise ValueError("seed must be None or an integer in [0, 2**63).")
    for value in (preserve_aspect_ratio, enable_vae_tiling, show_videos, download_text_encoder):
        if type(value) is not bool:
            raise ValueError("Aspect ratio, tiling, display, and download settings must be booleans.")
    for value in (sampler_name, noise_schedule):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("Select a sampler and noise schedule.")
    paths = {
        "checkpoint": resolve_path(checkpoint, family_dir / "checkpoints"),
        "vae": resolve_path(vae_weights, family_dir),
        "input": resolve_path(input_image, project_root),
        "text_encoder": resolve_path(text_encoder_weights, family_dir / "text_encoders"),
        "output": resolve_path(output_dir, project_root),
    }
    for name in ("checkpoint", "vae", "input"):
        if not paths[name].is_file():
            raise FileNotFoundError(f"{name} file not found: {paths[name]}")
    header = safetensors_header(paths["checkpoint"])
    config = json.loads(header.get("__metadata__", {}).get("config", "{}"))
    if config.get("transformer", {}).get("image_model") != "minimax_h3":
        raise ValueError("Select a ComfyUI MiniMax H3 checkpoint with image_model=minimax_h3 metadata.")
    vae_header = safetensors_header(paths["vae"])
    if "latents_mean" not in vae_header or not any(
        k.startswith(("decoder.transformer_blocks.", "decoder.blocks.")) for k in vae_header
    ):
        raise ValueError("Select the MiniMax H3 video VAE, not a WAN or audio VAE.")
    if not paths["text_encoder"].is_file():
        if not download_text_encoder:
            raise FileNotFoundError(f"MiniMax H3 Qwen3-VL text encoder not found: {paths['text_encoder']}")
        # Automatic downloads are restricted to documented H3 encoders.
        allowed = {f"qwen3vl_32b_minimax_h3_{kind}.safetensors"
                   for kind in ("bf16", "int8_convrot", "nvfp4_awq")}
        if str(text_encoder_weights) not in allowed:
            raise ValueError("For a custom text encoder path, supply the local file first.")
    if not isinstance(loras, (list, tuple)):
        raise ValueError("loras must be a list of {path, strength} entries.")
    selected, seen = [], set()
    for item in loras:
        if not isinstance(item, dict) or set(item) != {"path", "strength"}:
            raise ValueError("Each LoRA needs exactly path and strength.")
        strength = item["strength"]
        if type(strength) not in (int, float) or not math.isfinite(strength) or strength < 0:
            raise ValueError("LoRA strength must be a finite number >= 0.")
        path = resolve_path(item["path"], family_dir / "loras")
        if not path.is_file() or path.suffix.lower() != ".safetensors":
            raise FileNotFoundError(f"LoRA .safetensors file not found: {path}")
        if path in seen:
            raise ValueError(f"Duplicate LoRA: {path}")
        seen.add(path)
        if strength:
            selected.append({"path": path, "strength": float(strength)})
    return paths, selected


def stack_loras(base_model, selected):
    # Always start from the unpatched base: reruns cannot accumulate old adapters.
    model = base_model
    for item in selected:
        weights, metadata = comfy.utils.load_torch_file(
            str(item["path"]), safe_load=True, return_metadata=True,
        )
        previous_count = sum(len(patches) for patches in model.patches.values())
        patched, _ = comfy.sd.load_lora_for_models(
            model, None, weights, item["strength"], 0.0, lora_metadata=metadata,
        )
        del weights
        if sum(len(patches) for patches in patched.patches.values()) <= previous_count:
            raise ValueError(f"LoRA did not match any H3 model weights: {item['path']}")
        model = patched
    return model

def protect_quantized_transfers(root):
    """Make this model's quantized parameter conversions safe across inference modes."""
    from types import MethodType

    def guarded_apply(original):
        def apply(module, fn, recurse=True):
            # Same-device .to() can return a normal wrapper unchanged. Re-wrapping
            # it as a Parameter inside inference_mode breaks its version counter.
            # Disable inference mode for the conversion only, including nested callers.
            with torch.inference_mode(False), torch.no_grad():
                return original(fn, recurse=recurse)
        return apply

    protected = 0
    for module in root.modules():
        if getattr(module, "_minimax_transfer_guard", False):
            continue
        if any(hasattr(parameter, "__tensor_flatten__") for parameter in module.parameters(recurse=False)):
            module._apply = MethodType(guarded_apply(module._apply), module)
            module._minimax_transfer_guard = True
            protected += 1
    return protected


def protect_cached_transfers(cache):
    """Protect cached components before ComfyUI attempts to unload or reuse them."""
    protected = 0
    for name, attribute in (("base", "model"), ("clip", "cond_stage_model"), ("vae", "first_stage_model")):
        component = cache.get(name)
        if component is not None:
            protected += protect_quantized_transfers(getattr(component, attribute))
    return protected


## Generate and save videos

Run the cell below after helpers. Prompts and variations run sequentially. `saved_paths` contains the latest MP4 paths. Each result also saves JSON metadata and the exact starting frame in `Videos/<prompt>/`; filenames are unique across reruns.

Exports are silent, matching the original notebook. H3 still samples its joint video/audio latent internally; audio decoding/export would additionally need an audio VAE. The included video VAE is used for conditioning and decoding, with optional tiled decoding. ComfyUI manages model placement/offloading; this notebook does not force all components into GPU memory.

References: [official ComfyUI H3 guide](https://docs.comfy.org/tutorials/video/minimax/minimax-h3), [native conditioning and frame rules](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_extras/nodes_minimax_h3.py), [official I2V workflow](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_minimax_h3_i2v.json).

Generation disables inference mode and uses `torch.no_grad()`. A per-model guard also runs quantized parameter transfers outside inference mode, including offload calls made by nested ComfyUI code. This handles the known [ComfyUI quantized-wrapper issue](https://github.com/Comfy-Org/ComfyUI/issues/15733), including INT8 embedding tables in the NVFP4 encoder. The guard applies to these loaded model instances and does not edit ComfyUI or PyTorch files.

After updating, reload the notebook from disk and restart the kernel, then run settings → helpers → generation (setup is not needed again). The generation cell must print `MiniMax runtime: guarded quantized transfers v2`; if that line is absent, the editor is still executing an older cell.


In [10]:
from dotenv import load_dotenv
from IPython.display import Video, display
from PIL import Image, ImageOps
import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
import torch

print("MiniMax runtime: guarded quantized transfers v2")

# Validate all local assets before downloads or model loading.
paths, selected_loras = validate_settings()
with Image.open(paths["input"]) as opened:
    source_image = ImageOps.exif_transpose(opened).convert("RGB")
input_sha256 = hashlib.sha256(paths["input"].read_bytes()).hexdigest()
if preserve_aspect_ratio:
    ratio = source_image.width / source_image.height
    video_width = max(32, int(math.sqrt(width * height * ratio)) // 32 * 32)
    video_height = max(32, int(math.sqrt(width * height / ratio)) // 32 * 32)
    conditioning_image = source_image.resize((video_width, video_height), Image.Resampling.LANCZOS)
else:
    video_width, video_height = width, height
    conditioning_image = ImageOps.fit(source_image, (width, height), method=Image.Resampling.LANCZOS)
subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-version"], check=True, capture_output=True, timeout=10)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the Thor-compatible CUDA/PyTorch kernel.")
if not (comfyui_dir / "comfy_extras" / "nodes_minimax_h3.py").is_file():
    raise FileNotFoundError("Run the setup cell first to install ComfyUI with native H3 support.")

# Import the core runtime directly; no ComfyUI web server or custom nodes are required.
if str(comfyui_dir) not in sys.path:
    sys.path.insert(0, str(comfyui_dir))
import comfy.options
comfy.options.enable_args_parsing(False)  # Do not parse Jupyter's command-line flags.
import comfy.sd
import comfy.utils
import comfy.samplers
import comfy.model_management
from comfy_extras.nodes_minimax_h3 import MiniMaxH3ImageToVideo
from comfy_extras.nodes_custom_sampler import (
    BasicGuider, BasicScheduler, KSamplerSelect, RandomNoise, SamplerCustomAdvanced,
)
from nodes import VAEDecode, VAEDecodeTiled

if sampler_name not in comfy.samplers.SAMPLER_NAMES or noise_schedule not in comfy.samplers.SCHEDULER_NAMES:
    raise ValueError("Selected sampler/schedule is unavailable in this ComfyUI checkout.")
load_dotenv(project_root / ".env")
cache_root = resolve_path(os.getenv("CACHE_LOCATION") or "Models", project_root)
if not paths["text_encoder"].is_file():
    from huggingface_hub import hf_hub_download
    print("Downloading the required MiniMax H3 Qwen3-VL encoder (several GB on first run).")
    paths["text_encoder"] = Path(hf_hub_download(
        repo_id="Comfy-Org/MiniMax-H3", filename=f"text_encoders/{text_encoder_weights}",
        cache_dir=str(cache_root / "hub"), token=os.getenv("HUGGINGFACE_TOKEN") or None,
    ))

try:
    comfy_revision = subprocess.check_output(
        ["git", "-C", str(comfyui_dir), "rev-parse", "HEAD"], text=True,
    ).strip()
except (OSError, subprocess.SubprocessError):
    comfy_revision = "unknown"
protect_cached_transfers(globals().get("_minimax_cache", {}))
model_key = tuple(fingerprint(paths[k]) for k in ("checkpoint", "vae", "text_encoder"))
if globals().get("_minimax_cache", {}).get("key") != model_key:
    _minimax_cache = {}  # Release every cached model reference before a different checkpoint.
    for name in ("model", "base_model", "clip", "vae", "guider", "positive"):
        globals().pop(name, None)
    comfy.model_management.unload_all_models()
    gc.collect()
    torch.cuda.empty_cache()
    # Only publish a complete cache after every component loads successfully.
    base_model = comfy.sd.load_diffusion_model(str(paths["checkpoint"]))
    protect_quantized_transfers(base_model.model)
    clip = comfy.sd.load_clip([str(paths["text_encoder"])], clip_type=comfy.sd.CLIPType.MINIMAX)
    protect_quantized_transfers(clip.cond_stage_model)
    vae_state, vae_metadata = comfy.utils.load_torch_file(str(paths["vae"]), return_metadata=True)
    try:
        vae = comfy.sd.VAE(sd=vae_state, metadata=vae_metadata)
        protect_quantized_transfers(vae.first_stage_model)
    finally:
        del vae_state
    _minimax_cache = {"key": model_key, "base": base_model, "clip": clip, "vae": vae}
else:
    print("Reusing the loaded MiniMax H3 components.")
base_model, clip, vae = (_minimax_cache[k] for k in ("base", "clip", "vae"))
lora_key = tuple((fingerprint(item["path"]), item["strength"]) for item in selected_loras)
if _minimax_cache.get("lora_key") != lora_key:
    # Remove the old patched clone before loading a changed stack. The base is never fused.
    _minimax_cache.pop("model", None)
    _minimax_cache.pop("lora_key", None)
    for name in ("model", "guider"):
        globals().pop(name, None)
    model = stack_loras(base_model, selected_loras)
    _minimax_cache.update(model=model, lora_key=lora_key)
model = _minimax_cache["model"]
lora_metadata = [{"path": str(item["path"]), "strength": item["strength"],
                  "fingerprint": fingerprint(item["path"])} for item in selected_loras]
print(f"Checkpoint: {paths['checkpoint']}\nVAE: {paths['vae']}\nLoRAs: {lora_metadata}")
print(f"Output: {video_width}x{video_height}, {num_frames} frames at {fps} fps")
print(f"GPU: {torch.cuda.get_device_name(0)} | ComfyUI: {comfy_revision}")

paths["output"].mkdir(parents=True, exist_ok=True)
saved_paths = []
image_tensor = torch.from_numpy(np.array(conditioning_image).astype(np.float32) / 255.0).unsqueeze(0)
unique_prompts = list(dict.fromkeys(prompts))
torch.cuda.reset_peak_memory_stats()
# ComfyUI swaps/offloads quantized Parameters while encoding, sampling, and decoding.
# Keep ordinary tensors with version counters so later model transfers remain valid.
with torch.inference_mode(False), torch.no_grad():
    for prompt_index, prompt in enumerate(unique_prompts):
        slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60].rstrip("-") or "prompt"
        prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()[:16]
        prompt_dir = paths["output"] / f"{slug}-{prompt_hash}"
        prompt_dir.mkdir(parents=True, exist_ok=True)
        (prompt_dir / "prompt.txt").write_text(prompt, encoding="utf-8")
        # H3 builds joint video/audio latents even when exporting a silent video.
        positive, latent = MiniMaxH3ImageToVideo.execute(
            clip=clip, vae=vae, prompt=prompt, width=video_width, height=video_height,
            length=num_frames, first_frame=image_tensor,
        ).result
        for video_index in range(num_videos_per_prompt):
            video_seed = (secrets.randbelow(2**63) if seed is None else
                          (seed + prompt_index * num_videos_per_prompt + video_index) % 2**63)
            print(f"Prompt {prompt_index + 1}/{len(unique_prompts)}, variation {video_index + 1}: seed {video_seed}")
            guider = BasicGuider.execute(model=model, conditioning=positive)[0]
            sampler = KSamplerSelect.execute(sampler_name=sampler_name)[0]
            sigmas = BasicScheduler.execute(model=model, scheduler=noise_schedule,
                                            steps=num_inference_steps, denoise=1.0)[0]
            noise = RandomNoise.execute(noise_seed=video_seed)[0]
            torch.cuda.synchronize()
            start = perf_counter()
            sampled = SamplerCustomAdvanced.execute(
                noise=noise, guider=guider, sampler=sampler, sigmas=sigmas, latent_image=latent,
            )[0]
            # Core VAEDecode selects the video member of H3's joint latent.
            if enable_vae_tiling:
                frames = VAEDecodeTiled().decode(vae=vae, samples=sampled, tile_size=512,
                                                overlap=64, temporal_size=64, temporal_overlap=8)[0]
            else:
                frames = VAEDecode().decode(vae=vae, samples=sampled)[0]
            torch.cuda.synchronize()
            elapsed = perf_counter() - start
            if len(frames) != num_frames:
                raise RuntimeError(f"Expected {num_frames} decoded frames, got {len(frames)}.")
            output_path = prompt_dir / f"minimax-h3-i2v-seed-{video_seed}-{uuid4().hex}.mp4"
            # Stream uint8 frames to FFmpeg instead of duplicating the whole decoded video.
            with imageio.get_writer(str(output_path), fps=fps, codec="libx264", macro_block_size=1) as writer:
                for frame in frames:
                    writer.append_data((frame.detach().float().cpu().numpy().clip(0, 1) * 255).round().astype(np.uint8))
            del frame, frames, sampled, guider
            conditioning_path = output_path.with_suffix(".input.png")
            conditioning_image.save(conditioning_path)
            metadata = {
                "model_family": "MiniMax", "architecture": "minimax_h3", "mode": "fl2va",
                "checkpoint": str(paths["checkpoint"]), "vae": str(paths["vae"]),
                "text_encoder": str(paths["text_encoder"]), "asset_fingerprints": model_key,
                "comfyui_revision": comfy_revision, "torch_version": torch.__version__,
                "prompt": prompt, "seed": video_seed, "loras": lora_metadata,
                "input_image": str(paths["input"]), "input_sha256": input_sha256,
                "input_size": list(source_image.size), "conditioning_image": conditioning_path.name,
                "requested_size": [width, height], "width": video_width, "height": video_height,
                "preserve_aspect_ratio": preserve_aspect_ratio, "num_frames": num_frames,
                "fps": fps, "duration_seconds": num_frames / fps, "audio_exported": False,
                "num_inference_steps": num_inference_steps, "sampler": sampler_name,
                "noise_schedule": noise_schedule, "guider": "BasicGuider", "cfg": 1.0,
                "enable_vae_tiling": enable_vae_tiling, "generation_seconds": elapsed,
            }
            output_path.with_suffix(".json").write_text(
                json.dumps(metadata, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8",
            )
            saved_paths.append(output_path)
            print(f"Saved {output_path} | Generation: {elapsed:.1f}s")
            if show_videos:
                display(Video(filename=str(output_path), embed=True, html_attributes="controls loop"))
        del positive, latent
print(f"Saved {len(saved_paths)} videos. Peak CUDA allocation: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GiB")


MiniMax runtime: guarded quantized transfers v2
Reusing the loaded MiniMax H3 components.
Checkpoint: /home/malneyugnfl/SoftwareDev/Image_Generator/Models/Model_Families/MiniMax/checkpoints/minimaxH3_comfy_3086249.safetensors
VAE: /home/malneyugnfl/SoftwareDev/Image_Generator/Models/Model_Families/MiniMax/vae/minimax_h3_video_vae_fp16.safetensors
LoRAs: [{'path': '/home/malneyugnfl/SoftwareDev/Image_Generator/Models/Model_Families/MiniMax/loras/Minimax_H3_Unlocked_V2.1.safetensors', 'strength': 1.0, 'fingerprint': ('/home/malneyugnfl/SoftwareDev/Image_Generator/Models/Model_Families/MiniMax/loras/Minimax_H3_Unlocked_V2.1.safetensors', 298257970, 1793315461146904654)}]
Output: 832x448, 124 frames at 24 fps
GPU: NVIDIA Thor | ComfyUI: fd7639f477050e5c25b68fc38a9c2a6b6d988211
Prompt 1/1, variation 1: seed 3474740201494911521


  0%|          | 0/20 [00:00<?, ?it/s]

Saved /home/malneyugnfl/SoftwareDev/Image_Generator/Videos/the-camera-slowly-moves-forward-a-gentle-breeze-moves-throug-b2d24b7cddf7ba47/minimax-h3-i2v-seed-3474740201494911521-a23fbbea5e1743f2a56fd04f9063759a.mp4 | Generation: 445.1s


Saved 1 videos. Peak CUDA allocation: 6.0 GiB
